# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook walks through loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described with a Croissant schema, accessible at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and explore basic dataset description with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL (Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# .metadata is a DatasetMetadata object, which allows .to_json(), but should not be subscripted directly
metadata_json = dataset.metadata.to_json()
print(f"Dataset Name: {metadata_json.get('name', '')}\nDescription: {metadata_json.get('description', '')}")

## 2. Data Overview
Review available record sets and fields, referencing all entities by their `@id`.

In [ ]:
# List available record sets, their fields and columns by @id
meta = dataset.metadata.to_json()
record_sets = meta.get('recordSet', [])

if not record_sets:
    print('No record sets found in the Croissant metadata.')
else:
    print('RecordSets:')
    for rs in record_sets:
        rs_id = rs.get('@id', None)
        rs_name = rs.get('name', '')
        print(f"- RecordSet @id: {rs_id}, name: {rs_name}")
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        print('  Fields:')
        for field in fields:
            field_id = field.get('@id', '')
            field_name = field.get('name', '')
            print(f"    - Field @id: {field_id}, name: {field_name}")
        if 'column' in rs:
            cols = rs.get('column', [])
            if isinstance(cols, dict):
                cols = [cols]
            print('  Columns:')
            for col in cols:
                col_id = col.get('@id', '')
                col_name = col.get('name', '')
                print(f"    - Column @id: {col_id}, name: {col_name}")

# As a demonstration, show a few raw records for the first record set, if present.
if record_sets:
    rs_id = record_sets[0].get('@id')
    print(f"\nSample records from RecordSet {rs_id}:")
    try:
        for i, record in enumerate(dataset.records(record_set=rs_id)):
            print(record)
            if i >= 2:
                break
    except Exception as e:
        print('Could not fetch records:', e)

## 3. Data Extraction
Load all tabular data from each record set into pandas DataFrames. Reference all entities by their `@id`.

In [ ]:
# Collect all record set @ids
record_sets = [rs.get('@id') for rs in dataset.metadata.to_json().get('recordSet', [])]
print('Available record set @ids:', record_sets)

dataframes = {}
for rs_id in record_sets:
    try:
        records_list = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records_list)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records for RecordSet {rs_id}. Columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Could not load records for {rs_id}: {e}")

# For demonstration, select the first record set
main_recordset = None
if record_sets:
    main_recordset = record_sets[0]
    print(f"\nFirst few rows for RecordSet {main_recordset}:")
    display(dataframes[main_recordset].head())

## 4. Exploratory Data Analysis (EDA)
We'll filter and normalize a numeric field, and optionally group data if a categorical field is available. All column references will use their Croissant `@id` values, where applicable.

In [ ]:
# Identify a numeric field (by @id) -- list possible numeric columns
import numpy as np
df = dataframes[main_recordset]
numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
print('Numeric fields detected:', numeric_candidates)

# If there are numeric fields, pick the first for demonstration
if numeric_candidates:
    numeric_field = numeric_candidates[0]
    print('Using numeric field (by @id or column name):', numeric_field)
    threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 0
    print(f"Using threshold: {threshold:.2f}\n")
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field]-filtered_df[numeric_field].mean())/filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} values:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Optionally group by a categorical field
    categorical_candidates = df.select_dtypes(include=['object']).columns
    group_field = None
    for col in categorical_candidates:
        if col != numeric_field and df[col].nunique() > 1 and df[col].nunique() < len(df)/2:
            group_field = col
            break
    
    if group_field:
        print(f"Grouping by field: {group_field}\n")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped mean {numeric_field} by {group_field}:")
        display(grouped_df.head())
else:
    print('No numeric fields detected in the record set.')

## 5. Visualization
Let's plot the distribution of the main numeric field using matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_candidates:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30)
    plt.xlabel(numeric_field)
    plt.title(f"Distribution of {numeric_field} (by @id or column name)")
    plt.show()
    
    if group_field:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()
else:
    print('No numeric field to visualize.')

## 6. Conclusion
We have explored the FAIR² dataset about extracellular vesicle protein mass spectrometry using `mlcroissant`, referencing key entities by their Croissant `@id`. This notebook demonstrated metadata inspection, record set loading, numeric field filtering and normalization, grouped analysis, and visualization—all useful for beginning scientific data exploration.

Next steps could include a deeper dive into specific protein abundance patterns, downstream statistical testing, or export for integration with domain-specific bioinformatics pipelines.